In [1]:
import numpy as np

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# ============================================================
# FUNCTION 4 - WEEK 9 BAYESIAN OPTIMISATION
# Run from inside the week9/ folder
# ============================================================

# ------------------------------------------------------------
# 1. Load cumulative Week 9 data
# ------------------------------------------------------------

X = np.load("function4/initial_inputs.npy")
Y = np.load("function4/initial_outputs.npy").reshape(-1)

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nCurrent best:")
print(best_x, "->", best_y)

print("\nY range:")
print("min =", Y.min())
print("max =", Y.max())
print("std =", Y.std())


# ------------------------------------------------------------
# 2. Week 8 calibration check
# ------------------------------------------------------------
#
# Week 8 selected:
# [0.33524044, 0.44148150, 0.43811063, 0.42398236]
#
# GP prediction:
# mean ≈ 0.443847
# std  ≈ 0.260506
#
# Actual:
# 0.03393207485785554
# ------------------------------------------------------------

week8_pred_mean = 0.443847
week8_pred_std = 0.260506
week8_actual = 0.03393207485785554

week8_error = week8_actual - week8_pred_mean
week8_z_error = week8_error / week8_pred_std

print("\n================================")
print("WEEK 8 CALIBRATION CHECK")
print("================================")

print("Predicted mean:", week8_pred_mean)
print("Predicted std :", week8_pred_std)
print("Actual        :", week8_actual)

print("\nPrediction error:")
print(week8_error)

print("\nError / predicted std:")
print(week8_z_error)


# ------------------------------------------------------------
# 3. Fit ARD Matern GP
# ------------------------------------------------------------

kernel = (
    ConstantKernel(
        1.0,
        constant_value_bounds=(1e-3, 1e3)
    )
    *
    Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    +
    WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\n================================")
print("GP FIT")
print("================================")

print("\nFitted kernel:")
print(gp.kernel_)

lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1.0 / lengthscales
relative_sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:")
print(lengthscales)

print("\nNormalised inverse-lengthscale sensitivity:")
print(relative_sensitivity)


# ------------------------------------------------------------
# 4. Expected Improvement
# ------------------------------------------------------------

def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    valid = sigma > 1e-12

    Z = np.zeros_like(mu)
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        +
        sigma[valid] * norm.pdf(Z[valid])
    )

    return EI


# ------------------------------------------------------------
# 5. Candidate generation
# ------------------------------------------------------------

rng = np.random.default_rng(42)

local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal widths:", local_scale)
print("Wide widths:", wide_scale)

local_candidates = (
    best_x
    + rng.normal(
        0,
        local_scale,
        size=(80000, 4)
    )
)

wide_candidates = (
    best_x
    + rng.normal(
        0,
        wide_scale,
        size=(60000, 4)
    )
)

global_candidates = rng.uniform(
    0,
    1,
    size=(100000, 4)
)

local_candidates = np.clip(
    local_candidates,
    0,
    1
)

wide_candidates = np.clip(
    wide_candidates,
    0,
    1
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

# Remove near-duplicates

tree = cKDTree(X)

distance, _ = tree.query(
    candidates,
    k=1
)

candidates = candidates[
    distance > 0.008
]

print("\nCandidates after duplicate filtering:")
print(len(candidates))


# ------------------------------------------------------------
# 6. GP predictions
# ------------------------------------------------------------

mu, sigma = gp.predict(
    candidates,
    return_std=True
)


# ------------------------------------------------------------
# 7. Primary EI
# ------------------------------------------------------------

EI = expected_improvement(
    mu,
    sigma,
    best_y,
    xi=0.0
)

ei_idx = np.argmax(EI)

print("\n================================")
print("PRIMARY EI")
print("================================")

print("candidate =", candidates[ei_idx])
print("mean =", mu[ei_idx])
print("std =", sigma[ei_idx])
print("EI =", EI[ei_idx])


# ------------------------------------------------------------
# 8. EI sensitivity
# ------------------------------------------------------------

y_scale = np.std(Y)

xi_values = [
    0.0,
    0.01 * y_scale,
    0.05 * y_scale,
    0.10 * y_scale
]

print("\n================================")
print("EI SENSITIVITY")
print("================================\n")

for xi in xi_values:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        "xi =", f"{xi:.6e}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


# ------------------------------------------------------------
# 9. Highest predicted mean
# ------------------------------------------------------------

mean_idx = np.argmax(mu)

print("\n================================")
print("HIGHEST PREDICTED MEAN")
print("================================")

print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


# ------------------------------------------------------------
# 10. UCB diagnostics
# ------------------------------------------------------------

print("\n================================")
print("UCB DIAGNOSTICS")
print("================================\n")

for beta in [0.1, 0.25, 0.5, 1.0]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )

X shape: (38, 4)
Y shape: (38,)

Current best:
[0.357812 0.420904 0.424244 0.430762] -> 0.601059871344695

Y range:
min = -32.625660215962455
max = 0.601059871344695
std = 9.412927258828919

WEEK 8 CALIBRATION CHECK
Predicted mean: 0.443847
Predicted std : 0.260506
Actual        : 0.03393207485785554

Prediction error:
-0.40991492514214445

Error / predicted std:
-1.573533527604525

GP FIT

Fitted kernel:
2.55**2 * Matern(length_scale=[1.65, 1.4, 1.35, 1.44], nu=2.5) + WhiteKernel(noise_level=0.000637)

ARD lengthscales:
[1.65425907 1.39659923 1.34731381 1.44484013]

Normalised inverse-lengthscale sensitivity:
[0.2194304  0.25991331 0.26942107 0.25123522]

Local widths: [0.1 0.1 0.1 0.1]
Wide widths: [0.2 0.2 0.2 0.2]

Candidates after duplicate filtering:
239997

PRIMARY EI
candidate = [0.37144533 0.41139313 0.42283176 0.4299074 ]
mean = 0.3466559652203429
std = 0.26388889081989814
EI = 0.02353261018544474

EI SENSITIVITY

xi = 0.000000e+00 
 candidate = [0.37144533 0.41139313 0.42283

In [2]:
# ============================================================
# FINAL FUNCTION 4 - WEEK 9 SELECTION
# ============================================================
#
# Week 8 produced a calibration error of approximately -1.57
# predictive standard deviations, so we avoid aggressive
# exploration this week.
#
# The highest GP mean and every tested UCB beta identify
# exactly the same candidate, very close to the proven
# incumbent.
#
# We therefore use this stable local candidate rather than
# moving toward a more speculative high-uncertainty region.

mean_idx = np.argmax(mu)

week9_candidate = candidates[mean_idx]

print("Week 9 Function 4 candidate:")
print(week9_candidate)

print("\nPredicted mean:")
print(mu[mean_idx])

print("\nPredicted std:")
print(sigma[mean_idx])

portal = "-".join(
    f"{x:.6f}"
    for x in week9_candidate
)

print("\nPortal format:")
print(portal)

Week 9 Function 4 candidate:
[0.36198508 0.41201044 0.42143681 0.42772469]

Predicted mean:
0.3502398468525616

Predicted std:
0.26100386108191

Portal format:
0.361985-0.412010-0.421437-0.427725
